## Tracking of a Mosquito-path dataset optimised over a parameter grid, using likelihoods

In [ ]:
## Importing data and converting to a track
from datetime import datetime, timedelta
import os
import pandas as pd
from stonesoup.types.detection import Clutter, Detection
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from stonesoup.models.base_driver import NoiseCase
from stonesoup.models.driver import AlphaStableNSMDriver 
from stonesoup.models.transition.levy_linear import LevyLangevin, CombinedLinearLevyTransitionModel
from stonesoup.models.measurement.linear import LinearGaussian
import numpy as np
from scipy.stats import uniform

# Define Predictor, Resampler, and Updater
from stonesoup.predictor.particle import MarginalisedParticlePredictor
from stonesoup.resampler.particle import SystematicResampler
from stonesoup.types.track import Track
from stonesoup.updater.particle import MarginalisedParticleUpdater
# Particle Initialization
from scipy.stats import multivariate_normal
from stonesoup.types.numeric import Probability  # Similar to a float type
from stonesoup.types.state import MarginalisedParticleState
from stonesoup.types.array import CovarianceMatrices, StateVector, StateVectors
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.models.transition.linear import ConstantVelocity, RandomWalk
from stonesoup.types.state import GaussianState
from stonesoup.updater.kalman import KalmanUpdater
import os, numpy as np, pandas as pd
from datetime   import timedelta, datetime
from functools  import lru_cache
from scipy.stats      import lognorm
from scipy.optimize   import minimize
from scipy.special     import logsumexp
from numpy.random      import default_rng

from stonesoup.types.state      import GaussianState, MarginalisedParticleState
from stonesoup.types.array      import StateVectors, CovarianceMatrices
from stonesoup.types.detection  import Detection, TrueDetection
from stonesoup.types.track      import Track
from stonesoup.models.measurement.linear import LinearGaussian
from stonesoup.types.numeric    import Probability

In [ ]:
############################################################
# 0.  Imports  +  global constants used everywhere
############################################################

# ---------- generic SMC settings ----------
num_particles         = 500            # per-object MPF particles
P_D                    = 0.90           # detection prob.
KAPPA_Z                = 1e-5           # clutter spatial density
SEED                   = 1
rng                    = default_rng(SEED)
MAX_CLUTTER=2

# ---------- folders & file list ----------
# Load the CSV file- With thanks to Chloe Chung for the data!
folder = r"TrackedDatasets\mosquito data"
files  = [
    "fly_1",
    "fly_2",
    "mosquito_1",
    "mosquito_2",
    "mosquito_3_with_velocity",
    "mosquito_4_with_velocity"
]

# process only the first N lines of every CSV
optimise_T   = 100
filter_T     = 100
num_steps    = min(optimise_T, filter_T)

############################################################
# 1–3.  Load CSV  ✧  empirical σᵉ  ✧  detections
############################################################
datasets = {}                                      # name → info-dict

for fname in files:
    df  = pd.read_csv(os.path.join(folder, fname + ".csv")).iloc[:num_steps]

    has_vel = {'vel_x', 'vel_y'}.issubset(df.columns)
    has_z   = 'z' in df.columns

        # ─── positions array  (state dimension) ─────────────────────
    if has_z:                                       # 3-D position
        pos   = df[['x', 'y', 'z']].to_numpy(float)          # (T,3)
        ndim, meas_map = 3, (0, 1, 2)                  # observe x,y,z
    else:                                           # 2-D position + latent vel
        pos   = df[['x', 'vel_x', 'y', 'vel_y']].to_numpy(float)   # (T,4)
        ndim, meas_map = 4, (0, 2)                      # observe x,y

    # statistics for clutter box
    x_arr, y_arr = pos[:, 0], pos[:, 2] if has_vel else pos[:, 1]
    x_std, y_std = np.std(x_arr), np.std(y_arr)
    if has_z:
        z_arr, z_std = pos[:, 2], np.std(pos[:, 2])

    # synthetic, monotone timeline (1 Hz)
    start_time  = datetime.now()
    ts   = [start_time + timedelta(seconds=k) for k in range(len(pos))]

    # ---- empirical measurement noise σₑ ---------------------------
    sigma_e = 0.5*x_std
    num_sigma2=3
    if num_sigma2==1:
        sigma2_grid=[sigma_e**2]
    else:
        sigma2_grid = np.logspace(np.log10(sigma_e**2) - 2,
                              np.log10(sigma_e**2), num_sigma2)
    

    # ---- measurement models & detections --------------------------
    meas_models = {
        s2: LinearGaussian(ndim_state=ndim,
                           mapping=meas_map,
                           noise_covar=s2 * np.eye(len(meas_map)))
        for s2 in sigma2_grid
    }
    all_measurements  = {s2: [] for s2 in sigma2_grid}
    truth = GroundTruthPath()

    for p, t in zip(pos, ts):
        state_vector=[[*p]]
        truth.append(GroundTruthState(state_vector=state_vector, timestamp=t))

    for s2, mdl in meas_models.items():
        
        for p, t in zip(pos, ts):
            state_vector=[[*p]]
            measurement_set=set()
            measurement = mdl.function(truth[t], noise=True)
            if np.random.rand() <= P_D:
                measurement_set.add(
                    TrueDetection(state_vector=measurement,
                                timestamp=t,
                                groundtruth_path=truth,
                                measurement_model=mdl
                                )
                )
            # -------- clutter generation --------
            kappa_z = 0.5 * (MAX_CLUTTER - 1) / ((4 * x_std) ** len(meas_map))
            n_clutter = rng.integers(MAX_CLUTTER)          # {0, …, MAX_CLUTTER-1}
            for _ in range(n_clutter):
                x = uniform.rvs(np.mean(x_arr) - 2 * x_std, 4 * x_std, random_state=rng)
                y = uniform.rvs(np.mean(y_arr) - 2 * y_std, 4 * y_std, random_state=rng)
                if has_z:
                    z = uniform.rvs(np.mean(z_arr) - 2 * z_std, 4 * z_std, random_state=rng)
                    c_vec = np.array([[x], [y], [z]])
                else:
                    c_vec = np.array([[x], [y]])
                measurement_set.add(Clutter(c_vec, timestamp=t,
                                            measurement_model=mdl))
            all_measurements[s2].append(measurement_set)
    # ---- pack everything -----------------------------------------
    datasets[fname] = dict(
        ts            = ts,
        dimensions    = (has_vel, has_z),
        meas_models   = meas_models,
        measurements  = all_measurements,
        groundtruth   = truth,
        sigma2_grid   = sigma2_grid,
        sigma2_e      = sigma_e**2
    )

print("⇒ Prepared datasets:", ", ".join(datasets))

In [ ]:
############################################################
# 4.  Prior states  (one Marginalised-Particle + one Gaussian per file)
############################################################
from stonesoup.types.state import GaussianState, MarginalisedParticleState
from stonesoup.types.array import StateVectors, CovarianceMatrices
from datetime import timedelta

# 4.  Priors  (one LP & one GP Track per CSV file)
# ===============================================
priors = {}                                     # file → (lp_prior , gp_prior)

for name, data in datasets.items():
    has_vel, has_z = data['dimensions']
    pos0 = data['groundtruth'][0].state_vector.flatten()      # (2,) or (3,)

    if has_vel:                               # ---------------- 4-D state
        μ0 = pos0               # x,vx,y,vy
        Σ0 = np.diag((0.01*np.abs(μ0) + 1e-6)**2)            # (4,4)
    elif has_z:                              # ---------------- 3-D state
        μ0 = pos0                                            # x,y,z
        Σ0 = np.diag((0.01*np.abs(μ0) + 1e-6)**2)            # (3,3)

    t0 = data['groundtruth'][0].timestamp - timedelta(microseconds=1)

    # --- LP particle prior -----------------------------------------
    pts = rng.multivariate_normal(μ0, Σ0, num_particles)     # (N,d)
    cov_stack = np.repeat(Σ0[:, :, None], num_particles, axis=2)  # (d,d,N)

    lp_prior = Track(MarginalisedParticleState(
        state_vector = StateVectors(pts.T),                  # (d,N)
        covariance   = CovarianceMatrices(cov_stack),        # (d,d,N)
        weight       = np.full(num_particles, 1/num_particles),
        timestamp    = t0
    ))

    # --- GP Gaussian prior -----------------------------------------
    gp_prior = Track(GaussianState(state_vector=μ0, covar=Σ0, timestamp=t0))

    priors[name] = (lp_prior, gp_prior)


############################################################
# 5.  Tiny helper : tolerant dictionary key look-up
############################################################
def close_key(dct, value, tol=1e-12):
    """Return the key in *dct* closest to *value* (within tol)."""
    keys = np.fromiter(dct, float)
    k    = keys[np.argmin(np.abs(keys - value))]
    if abs(k - value) > tol:
        raise KeyError(value)
    return float(k)


In [ ]:
############################################################
# 6.  Cached builders  (transition-model ➜ predictor + updaters)
############################################################
from functools import lru_cache

from stonesoup.measures.state import Mahalanobis

# — Gaussian side —
from stonesoup.dataassociator.neighbour import GlobalNearestNeighbour
from stonesoup.hypothesiser.distance import DistanceHypothesiser
from stonesoup.models.transition.linear import (
    RandomWalk, OrnsteinUhlenbeck, CombinedLinearGaussianTransitionModel)
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.updater.kalman   import KalmanUpdater

# — Lévy / particle side —
from stonesoup.models.driver               import AlphaStableNSMDriver
from stonesoup.models.transition.levy_linear import (
    LevyRandomWalk, LevyLangevin, CombinedLinearLevyTransitionModel)
from stonesoup.predictor.particle          import MarginalisedParticlePredictor
from stonesoup.updater.particle            import MarginalisedParticleUpdater
from stonesoup.resampler.particle          import SystematicResampler

RESAMPLER = SystematicResampler()           # single shared instance


@lru_cache(maxsize=None)
def build_gp(name: str, sigma_w2: float, theta: float):
    """
    Return (predictor, {σe² : updater}) for the Gaussian model attached
    to *name*.  • OU in 2-D (hidden velocity)  • RW in 2- or 3-D otherwise.
    """
    has_vel, has_z = datasets[name]['dimensions']

    # --- choose transition model ----------------------------------
    if has_vel:                  # x,y with latent velocity components
        tm = CombinedLinearGaussianTransitionModel(
                 [OrnsteinUhlenbeck(sigma_w2, theta)]*2)
    elif has_z:                  # 3-D random walk
        tm = CombinedLinearGaussianTransitionModel(
                 [RandomWalk(sigma_w2)]*3)
    else:                        # plain 2-D random walk
        tm = CombinedLinearGaussianTransitionModel(
                 [RandomWalk(sigma_w2)]*2)

    predictor = KalmanPredictor(tm)
    updaters  = {s2: KalmanUpdater(mdl)
                 for s2, mdl in datasets[name]['meas_models'].items()}
    data_associators = {s2: GlobalNearestNeighbour(
                                                DistanceHypothesiser(predictor, KalmanUpdater(mdl),
                                                measure=Mahalanobis(), 
                                                missed_distance=0.1)) 
                                                for s2, mdl in datasets[name]['meas_models'].items()}

    return data_associators, updaters


@lru_cache(maxsize=None)
def build_lp(name: str, sigma_w2: float, theta: float, alpha: float):
    """
    Return (predictor, {σe² : updater}) for the Lévy (MPF) model
    attached to *name*.  • Lévy-Langevin in 2-D with vel,
    • Lévy RW in 2- or 3-D otherwise.
    """
    driver = AlphaStableNSMDriver(mu_W=0.0,
                                  sigma_W2=sigma_w2,
                                  c=10.0,
                                  alpha=alpha,
                                  noise_case=NoiseCase(2))

    has_vel, has_z = datasets[name]['dimensions']

    if has_vel:
        tm = CombinedLinearLevyTransitionModel(
                 [LevyLangevin(driver,
                               noise_diff_coeff=sigma_w2**0.5,
                               damping_coeff=theta)]*2)
    elif has_z:
        tm = CombinedLinearLevyTransitionModel(
                 [LevyRandomWalk(driver, noise_diff_coeff=sigma_w2)]*3)
        
    predictor = MarginalisedParticlePredictor(tm)
    updaters  = {s2: MarginalisedParticleUpdater(mdl, RESAMPLER)
                 for s2, mdl in datasets[name]['meas_models'].items()}
    data_associators = {s2: GlobalNearestNeighbour(
                                            DistanceHypothesiser(predictor, MarginalisedParticleUpdater(mdl, RESAMPLER),
                                            measure=Mahalanobis(), 
                                            missed_distance=0.1)) 
                                            for s2, mdl in datasets[name]['meas_models'].items()}
    return data_associators, updaters


In [ ]:
###################################################################
# 7–9.  Per-file Nelder–Mead search for (σw², θ [, α])            #
###################################################################
from scipy.optimize import minimize
from scipy.special  import logsumexp
from stonesoup.types.hypothesis import SingleHypothesis

# ────────────────────────────────────────────────────────────────
#  helper: log-likelihood for ONE file & ONE hyper-parameter set
# ────────────────────────────────────────────────────────────────
def _filter_loglike(name, params, model='gp'):
    """Log p(y | θ) with σe² integrated-out (Monte-Carlo for LP)."""
    if model == 'gp':
        σw2, θ = params
        associators, updaters, = build_gp(name, σw2, θ)
        base_track     = priors[name][1]          # GP prior Track
    else:
        σw2, θ, α = params
        associators, updaters = build_lp(name, σw2, θ, α)
        base_track     = priors[name][0]          # LP prior Track

    ll_total = 0.0
    for s2, all_meas in datasets[name]['measurements'].items():        
        track   = Track(base_track[0])            # fresh copy
        associator=associators[s2]
        updater = updaters[s2]
        
        for t, dets in enumerate(all_meas[:optimise_T]):
            timestamp=datasets[name]['ts'][t]

            hypo= associator.associate(tracks=[track],
                                            detections=dets,
                                            timestamp=timestamp)[track]
            if hypo.measurement:
                det=hypo.measurement
                post       = updater.update(hypo)
                track.append(post)

                mp   = hypo.measurement_prediction
                y    = det.state_vector.flatten()
                if model == 'gp':                     # single Gaussian
                    μ = mp.state_vector.flatten()
                    Σ = mp.covar
                    diff = y - μ
                    Σinv = np.linalg.inv(Σ)
                    maha = diff @ Σinv @ diff
                    d    = y.size
                    sign, logdet = np.linalg.slogdet(Σ)
                    ll_total += -0.5*(maha + d*np.log(2*np.pi) + logdet)
                else:                                 # particle mixture
                    means = mp.state_vector           # (d,N)
                    covs  = mp.covariance             # (d,d,N)
                    d, N  = means.shape
                    ll_arr = np.empty(N)
                    for j in range(N):
                        diff  = y - means[:, j]
                        Σinv  = np.linalg.inv(covs[:, :, j])
                        maha  = diff @ Σinv @ diff
                        sign, logdet = np.linalg.slogdet(covs[:, :, j])
                        ll_arr[j] = -0.5*(maha + d*np.log(2*np.pi) + logdet)
                    ll_total += logsumexp(ll_arr) - np.log(N)
            else:
                track.append(hypo.prediction)
                ll_total += np.log(1-P_D)
    return ll_total


# ────────────────────────────────────────────────────────────────
#  dictionaries to hold arg-max results
# ────────────────────────────────────────────────────────────────
Best_gp_params = {}      # file → (σw2 , θ)
Best_lp_params = {}      # file → (σw2 , θ , α)

print("⏳  Optimising hyper-parameters (per file) …")

for name in datasets.keys():

    # --- convenience sigma2_e for initialisation ------------------
    init_σw2 = datasets[name]['sigma2_e']                                      # reasonable start
    init_θ  = 0.1
    init_α  = 1.7
    
    # -------------- objective wrappers for SciPy -----------------
    def obj_gp(x):
        σw2 = np.exp(x[0])
        θ   = 0.05+ 0.45/ (1.0 + np.exp(-x[1]))                # map ℝ → (0.05,0.5)
        print('iteration')
        return -_filter_loglike(name, (σw2, θ), model='gp')

    def obj_lp(x):
        σw2  = np.exp(x[0])
        θ   = 0.05+ 0.45/ (1.0 + np.exp(-x[1]))                # map ℝ → (0.05,0.5)
        α    = 0.5 + 1.4/(1+np.exp(-x[2]))               # (0.5,1.9)
        print('iteration')
        if abs(α-1.0) < 1e-3:   # avoid α=1 exactly
            α += 0.05
        return -_filter_loglike(name, (σw2, θ, α), model='lp')

    # ---------------- Nelder–Mead (Gaussian) ---------------------
    res_gp = minimize(obj_gp,
                      x0=[np.log(init_σw2), init_θ],
                      method='Nelder-Mead',
                      options={'maxiter': 5, 'disp': False})

    σw_best_gp = np.sqrt(np.exp(res_gp.x[0]))
    θ_best_gp  = 0.05+ 0.45/ (1.0+ np.exp(-res_gp.x[1]))
    Best_gp_params[name] = (σw_best_gp, θ_best_gp)

    print(f"  • {name:<24}   GP→ σw={σw_best_gp:.3f}, θ={θ_best_gp:.2f}  ")

    # ---------------- Nelder–Mead (Lévy) -------------------------
    res_lp = minimize(obj_lp,
                      x0=[np.log(init_σw2), init_θ, init_α],
                      method='Nelder-Mead',
                      options={'maxiter': 5, 'disp': False})

    σw_best_lp = np.sqrt(np.exp(res_lp.x[0]))
    θ_best_lp  = 0.05+ 0.45/ (1.0+ np.exp(-res_lp.x[1]))
    α_best_lp  = 0.5 + 1.4/(1+np.exp(-res_lp.x[2]))
    if abs(α_best_lp-1.0) < 1e-3:
        α_best_lp += 0.05
    Best_lp_params[name] = (σw_best_lp, θ_best_lp, α_best_lp)

    print(f"  Lévy→ σw={σw_best_lp:.3f}, θ={θ_best_lp:.2f}, α={α_best_lp:.2f}")

print("\n✔  Hyper-parameter optimisation finished.")


In [ ]:
############################################################
# 10–12.  Optimal-parameter filtering  ➜  HTML plots
############################################################
from copy import deepcopy
from pathlib import Path
from datetime import timedelta

from stonesoup.plotter import Plotterly, AnimatedPlotterly, Dimension
from stonesoup.smoother.particle import MarginalisedKalmanSmoother
from stonesoup.types.hypothesis import SingleHypothesis
from stonesoup.updater.kalman      import KalmanUpdater
from stonesoup.updater.particle    import MarginalisedParticleUpdater
from stonesoup.resampler.particle  import SystematicResampler

Optimal_lp_tracks, Optimal_gp_tracks = {}, {}

out_root = Path(r"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB"
                r"\PROJECT- Implementation of N-G TAs in SS framework\Mosquito")
out_root.mkdir(parents=True, exist_ok=True)

def _pretty(fig, xlab, ylab):
    fig.update_layout(width=1200, height=640, plot_bgcolor="white",
                      xaxis=dict(showgrid=True, gridcolor="lightgray",
                                 title=dict(text=xlab, font=dict(size=20))),
                      yaxis=dict(showgrid=True, gridcolor="lightgray",
                                 title=dict(text=ylab, font=dict(size=20))),
                      legend=dict(font=dict(size=15),
                                  bordercolor="Black", borderwidth=2))

for name, data in datasets.items():

    has_vel, has_z   = data['dimensions']
    truth_track      = data['groundtruth']

    # ---- 1.  pick one σe² slice (empirical nearest) ------------------

    σe2_emp      = np.median(data['sigma2_grid'])
    σe2_plot     = close_key(data['meas_models'], σe2_emp)
    meas_model   = data['meas_models'][σe2_plot]
    all_meas     = data['measurements'][σe2_plot]

    gp_upd_plot  = KalmanUpdater(meas_model)
    lp_upd_plot  = MarginalisedParticleUpdater(meas_model, SystematicResampler())

    # ---- 2.  build predictors from the *best* θ, α, σw ---------------
    σw2_gp, θ_gp               = Best_gp_params[name]
    σw2_lp, θ_lp, α_lp         = Best_lp_params[name]

    gp_associator_opt, gp_updater_opt = build_gp(name, σw2_gp, θ_gp)
    lp_associator_opt, lp_updater_opt= build_lp(name, σw2_lp, θ_lp, α_lp)

    # fresh priors
    lp_track = Track(deepcopy(priors[name][0]))
    gp_track = Track(deepcopy(priors[name][1]))

    # ---- 3.  one-pass filtering over *all* detections ----------------
    for t, dets in enumerate(all_meas[:filter_T]):
        timestamp=datasets[name]['ts'][t]

        # Lévy
        lp_hypo= lp_associator_opt[σe2_plot].associate(tracks=[lp_track],
                                        detections=dets,
                                        timestamp=timestamp)[lp_track]
        if lp_hypo.measurement:
            lp_post       = lp_updater_opt[σe2_plot].update(lp_hypo)
            lp_track.append(lp_post)
        else:
            lp_track.append(lp_hypo.prediction)
        # Gaussian
        gp_hypo= gp_associator_opt[σe2_plot].associate(tracks=[gp_track],
                                        detections=dets,
                                        timestamp=timestamp)[gp_track]
        if gp_hypo.measurement:
            gp_post       = gp_updater_opt[σe2_plot].update(gp_hypo)
            gp_track.append(gp_post)
        else:
            gp_track.append(gp_hypo.prediction)


    Optimal_lp_tracks[name], Optimal_gp_tracks[name] = lp_track, gp_track

    # -----------------------------------------------------------------
    # 4-A.  1-D x-coordinate panel
    # -----------------------------------------------------------------
    RTSsmoother=MarginalisedKalmanSmoother()
    RTS_track=RTSsmoother.smooth(lp_track[1:]) #[1:] means we exclude prior state (isn't a prediction or a post)

    p1 = Plotterly(dimension=Dimension.ONE)
    p1.plot_ground_truths(truth_track, [0], truths_label="True x",
                          line=dict(width=3))
    p1.plot_measurements(all_meas, [0], label="Meas x",marker=dict(size=8))
    p1.plot_tracks(lp_track, [0], track_label="Lévy",    uncertainty=True,
                   line=dict(width=3, color="purple"))
    p1.plot_tracks(gp_track, [0], track_label="Gaussian", uncertainty=True,
                   line=dict(width=3, color="green"))
    p1.plot_tracks(RTS_track, [0], track_label="Levy RTS", uncertainty=True,
                   line=dict(width=3, color='#B6E880'))
    _pretty(p1.fig, "Time","x-position")
    p1.fig.write_html(out_root / f"{name}_1D.html")

    # -----------------------------------------------------------------
    # 4-B.  2-D / 3-D animated trajectory (skip if pure-1D data)
    # -----------------------------------------------------------------
    T     = len(truth_track)
    times = [truth_track[0].timestamp + timedelta(seconds=k) for k in range(T)]
    if has_vel:
        mapping  = [0,2]
    else:
        mapping  = [0,1]


    p2 = AnimatedPlotterly(timesteps=times,tail_length=0.3)
    p2.plot_measurements(all_meas, mapping, label="Measurements")
    p2.plot_ground_truths(truth_track, mapping, truths_label="True path",
                    line=dict(width=3))
    p2.plot_tracks(lp_track, mapping, track_label="Lévy",    uncertainty=True,
                    line=dict(width=3, color="purple"))
    p2.plot_tracks(gp_track, mapping, track_label="Gaussian", uncertainty=True,
                    line=dict(width=3, color="green"))
    p1.plot_tracks(RTS_track, mapping, track_label="Levy RTS", uncertainty=True,
                   line=dict(width=3, color='#B6E880'))
    _pretty(p2.fig, "x-position","y-position")
    p2.fig.write_html(out_root / f"{name}_2Danimated.html")

    p2 = Plotterly(timesteps=times)
    p2.plot_measurements(all_meas, mapping, label="Measurements")
    p2.plot_ground_truths(truth_track, mapping, truths_label="True path",
                    line=dict(width=3))
    p2.plot_tracks(lp_track, mapping, track_label="Lévy",    uncertainty=True,
                    line=dict(width=3, color="purple"))
    p2.plot_tracks(gp_track, mapping, track_label="Gaussian", uncertainty=True,
                    line=dict(width=3, color="green"))
    p2.plot_tracks(RTS_track, mapping, track_label="Levy RTS", uncertainty=True,
                   line=dict(width=3, color='#B6E880'))
    _pretty(p2.fig, "x-position","y-position")
    p2.fig.write_html(out_root / f"{name}_2D.html")

    # -----------------------------------------------------------------
    # 4-A.  1-D x-velocity panel
    # -----------------------------------------------------------------
    if has_vel:
        p3 = Plotterly(dimension=Dimension.ONE, autosize=False)
        p3.plot_ground_truths(truth_track, [1], truths_label="True x",
                    line=dict(width=3))
        p3.plot_tracks(lp_track, [1], track_label="Lévy",    uncertainty=True,
                    line=dict(width=3, color="purple"))
        p3.plot_tracks(gp_track, [1], track_label="Gaussian", uncertainty=True,
                    line=dict(width=3,color="green"))
        _pretty(p3.fig,  "x-velocity","Time")
        p3.fig.write_html(out_root / f"{name}.html")

print("🎉  Filtering complete – HTML plots written to", out_root)
